# 04 — Modeling (PostgreSQL Pipeline)

**Sources**: `features.master_features` + `features.target_labels`

**Models**: Ridge/Lasso, RandomForest, XGBoost, LightGBM, CatBoost, Optuna+XGB, LSTM

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

PROJECT_ROOT = Path().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')
from src.data.storage.postgres_client import get_engine

MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PREDS_DIR  = PROJECT_ROOT / 'data' / 'predictions'
PREDS_DIR.mkdir(parents=True, exist_ok=True)

engine = get_engine()
print('[OK] Setup done')

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────
with engine.connect() as conn:
    df_feat = pd.read_sql(
        'SELECT * FROM features.master_features ORDER BY date',
        conn, index_col='date', parse_dates=['date']
    )
    df_tgt = pd.read_sql(
        """SELECT date, next_7_day_price
           FROM features.target_labels
           WHERE next_7_day_price IS NOT NULL ORDER BY date""",
        conn, index_col='date', parse_dates=['date']
    )

df = df_feat.join(df_tgt, how='inner')
print(f'[INFO] Combined shape: {df.shape}')
print(f'[INFO] Range: {df.index.min().date()} → {df.index.max().date()}')

In [ ]:
# ── Feature selection ──────────────────────────────────────────────────────
# Các cột target (không được dùng làm feature)
TARGET_COLS = [c for c in df.columns if c.startswith('next_')] + ['updated_at']

# Các cột raw OHLC hiện tại (potential leak nếu không cẩn thận)
RAW_PRICE_COLS = ['gold_open', 'gold_high', 'gold_low', 'gold_volume']

# FRED monthly hiện chưa có release-date/vintage nên loại khỏi backtest
PIT_UNSAFE = ['fed_funds_rate', 'us_interest_rate', 'us_inflation_yoy',
              'cpi', 'core_cpi', 'm2_money_supply', 'unemployment_rate']

EXCLUDE = TARGET_COLS + RAW_PRICE_COLS + PIT_UNSAFE
EXCLUDE = [c for c in EXCLUDE if c in df.columns]

TARGET = 'next_7_day_price'
feature_cols = [c for c in df.columns if c not in EXCLUDE]

data_clean = df[feature_cols + [TARGET]].dropna()
print(f'[INFO] Features: {len(feature_cols)} | Clean rows: {len(data_clean)}')

In [ ]:
# ── Walk-forward split (80/20 chronological) ──────────────────────────────
from sklearn.preprocessing import StandardScaler

n = len(data_clean)
split_idx = int(n * 0.80)

# Purge 7 observations để label t+7 của train không chạm test period
train_df = data_clean.iloc[:split_idx - 7]
test_df  = data_clean.iloc[split_idx:]

X_train, y_train = train_df[feature_cols].values, train_df[TARGET].values
X_test,  y_test  = test_df[feature_cols].values,  test_df[TARGET].values

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
joblib.dump(scaler, MODELS_DIR / 'scaler.joblib')

print(f'Train: {train_df.index.min().date()} → {train_df.index.max().date()} ({len(train_df)} rows)')
print(f'Test : {test_df.index.min().date()}  → {test_df.index.max().date()}  ({len(test_df)} rows)')

In [ ]:
# ── Evaluation helpers ─────────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results = []

def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f'[{name:18s}] MAE={mae:8.2f}  RMSE={rmse:8.2f}  R²={r2:.4f}  MAPE={mape:.2f}%')
    results.append({'model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape})
    return y_pred

In [ ]:
# ── 1. Baseline: Ridge & Lasso ─────────────────────────────────────────────
from sklearn.linear_model import Ridge, Lasso

for ModelClass, name in [(Ridge, 'Ridge'), (Lasso, 'Lasso')]:
    m = ModelClass(alpha=1.0, max_iter=10000)
    m.fit(X_train_sc, y_train)
    evaluate(name, y_test, m.predict(X_test_sc))
    joblib.dump(m, MODELS_DIR / f'{name.lower()}.joblib')
print('[OK] Ridge & Lasso done')

In [ ]:
# ── 2. Random Forest ───────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300, max_depth=10,
    min_samples_leaf=5, n_jobs=-1, random_state=42
)
rf.fit(X_train, y_train)
rf_pred = evaluate('RandomForest', y_test, rf.predict(X_test))
joblib.dump(rf, MODELS_DIR / 'random_forest.joblib')
print('[OK] RandomForest done')

In [ ]:
# ── 3. XGBoost ─────────────────────────────────────────────────────────────
try:
    import xgboost as xgb
    xgb_model = xgb.XGBRegressor(
        n_estimators=800, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0,
        n_jobs=-1, random_state=42, eval_metric='rmse'
    )
    xgb_model.fit(X_train, y_train)
    xgb_pred = evaluate('XGBoost', y_test, xgb_model.predict(X_test))
    xgb_model.save_model(str(MODELS_DIR / 'xgboost.json'))
    print('[OK] XGBoost done')
except ImportError:
    print('[SKIP] xgboost not installed')

In [ ]:
# ── 4. LightGBM ────────────────────────────────────────────────────────────
try:
    import lightgbm as lgb
    lgb_model = lgb.LGBMRegressor(
        n_estimators=800, learning_rate=0.05, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        n_jobs=-1, random_state=42, verbose=-1
    )
    lgb_model.fit(X_train, y_train)
    lgb_pred = evaluate('LightGBM', y_test, lgb_model.predict(X_test))
    lgb_model.booster_.save_model(str(MODELS_DIR / 'lightgbm.txt'))
    print('[OK] LightGBM done')
except ImportError:
    print('[SKIP] lightgbm not installed')

In [ ]:
# ── 5. CatBoost ────────────────────────────────────────────────────────────
try:
    from catboost import CatBoostRegressor
    cb_model = CatBoostRegressor(
        iterations=800, learning_rate=0.05, depth=6,
        l2_leaf_reg=3.0, random_seed=42,
        eval_metric='RMSE',
        verbose=100
    )
    cb_model.fit(X_train, y_train)
    cb_pred = evaluate('CatBoost', y_test, cb_model.predict(X_test))
    cb_model.save_model(str(MODELS_DIR / 'catboost.cbm'))
    print('[OK] CatBoost done')
except ImportError:
    print('[SKIP] catboost not installed')

In [ ]:
# ── 6. Optuna + XGBoost (AutoML Tuning) ───────────────────────────────────
try:
    import optuna
    import xgboost as xgb
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    from sklearn.model_selection import TimeSeriesSplit

    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 200, 1000),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'max_depth':        trial.suggest_int('max_depth', 3, 10),
            'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
            'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
            'n_jobs': -1, 'random_state': 42, 'verbosity': 0
        }
        fold_rmse = []
        tscv = TimeSeriesSplit(n_splits=3, gap=7)
        for train_idx, val_idx in tscv.split(X_train):
            m = xgb.XGBRegressor(**params)
            m.fit(X_train[train_idx], y_train[train_idx])
            pred = m.predict(X_train[val_idx])
            fold_rmse.append(np.sqrt(mean_squared_error(y_train[val_idx], pred)))
        return float(np.mean(fold_rmse))

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=50, show_progress_bar=True)

    print(f'\nBest RMSE: {study.best_value:.4f}')
    print(f'Best params: {study.best_params}')

    best_xgb = xgb.XGBRegressor(**study.best_params, n_jobs=-1, random_state=42, verbosity=0)
    best_xgb.fit(X_train, y_train)
    optuna_pred = evaluate('Optuna+XGB', y_test, best_xgb.predict(X_test))
    best_xgb.save_model(str(MODELS_DIR / 'xgboost_optuna.json'))
    print('[OK] Optuna+XGB done')
except ImportError:
    print('[SKIP] optuna/xgboost not installed')

In [ ]:
# ── 7. LSTM (Keras/TensorFlow) ─────────────────────────────────────────────
LOOKBACK = 30

def make_sequences(X, y, lookback):
    Xs, ys = [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i - lookback: i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    tf.random.set_seed(42)

    X_tr_seq, y_tr_seq = make_sequences(X_train_sc, y_train, LOOKBACK)
    X_te_seq, y_te_seq = make_sequences(X_test_sc,  y_test,  LOOKBACK)

    n_feat = X_tr_seq.shape[2]
    lstm_model = keras.Sequential([
        layers.Input(shape=(LOOKBACK, n_feat)),
        layers.LSTM(128, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(64),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1),
    ], name='LSTM_gold')
    lstm_model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])

    cbs = [
        keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, monitor='val_loss'),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=7, min_lr=1e-6)
    ]
    lstm_model.fit(
        X_tr_seq, y_tr_seq,
        validation_data=(X_te_seq, y_te_seq),
        epochs=100, batch_size=64, callbacks=cbs, verbose=1
    )
    lstm_pred = lstm_model.predict(X_te_seq).flatten()
    evaluate('LSTM', y_te_seq, lstm_pred)
    lstm_model.save(str(MODELS_DIR / 'lstm_gold.keras'))
    print('[OK] LSTM done')
except ImportError:
    print('[SKIP] tensorflow not installed')

In [ ]:
# ── Leaderboard & Visualization ────────────────────────────────────────────
results_df = pd.DataFrame(results).set_index('model').sort_values('RMSE')
print('\n=== Leaderboard (test set) ===')
print(results_df)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric in zip(axes, ['MAE', 'RMSE', 'MAPE']):
    results_df[metric].sort_values().plot.barh(ax=ax, color='steelblue')
    ax.set_title(f'Test {metric}')
    ax.set_xlabel(metric)
plt.suptitle('Holdout Comparison — Gold Close (t+7)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n[INFO] Holdout comparison only; canonical model selection uses CV in src/modeling/train.py')

In [ ]:
# ── Feature Importance (XGBoost / LightGBM) ───────────────────────────────
try:
    import xgboost as xgb
    fi = pd.Series(xgb_model.feature_importances_, index=feature_cols)
    fi = fi.nlargest(25)

    fig, ax = plt.subplots(figsize=(10, 8))
    fi.sort_values().plot.barh(ax=ax, color='steelblue')
    ax.set_title('XGBoost Feature Importance (Top 25)', fontsize=13)
    ax.set_xlabel('Importance Score')
    plt.tight_layout()
    plt.show()
except NameError:
    print('[SKIP] xgb_model not available')

In [ ]:
# ── Save predictions ───────────────────────────────────────────────────────
preds_dict = {'date': test_df.index, 'y_true': y_test}
# Add predictions from each model if available
try: preds_dict['xgboost'] = xgb_pred
except NameError: pass
try: preds_dict['lightgbm'] = lgb_pred
except NameError: pass
try: preds_dict['catboost'] = cb_pred
except NameError: pass

df_preds = pd.DataFrame(preds_dict).set_index('date')
df_preds.to_parquet(PREDS_DIR / 'cv_results.parquet')
results_df.to_parquet(PREDS_DIR / 'leaderboard.parquet')
print(f'[OK] Saved predictions → {PREDS_DIR}')
print(f'[OK] Saved leaderboard → {PREDS_DIR}')